In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv('final_internship_data (1).csv')

In [3]:
df.shape

(500000, 26)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 26 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   User ID            500000 non-null  object 
 1   User Name          500000 non-null  object 
 2   Driver Name        500000 non-null  object 
 3   Car Condition      500000 non-null  object 
 4   Weather            500000 non-null  object 
 5   Traffic Condition  500000 non-null  object 
 6   key                500000 non-null  object 
 7   fare_amount        500000 non-null  float64
 8   pickup_datetime    500000 non-null  object 
 9   pickup_longitude   500000 non-null  float64
 10  pickup_latitude    500000 non-null  float64
 11  dropoff_longitude  499995 non-null  float64
 12  dropoff_latitude   499995 non-null  float64
 13  passenger_count    500000 non-null  int64  
 14  hour               500000 non-null  int64  
 15  day                500000 non-null  int64  
 16  mo

In [5]:
df.head()

,User ID,User Name,Driver Name,Car Condition,Weather,Traffic Condition,key,fare_amount,pickup_datetime,pickup_longitude,...,month,weekday,year,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing
0,KHVrEVlD,Kimberly Adams,Amy Butler,Very Good,windy,Congested Traffic,2009-06-15 17:26:21.0000001,4.5,2009-06-15 17:26:21,-1.288826,...,6,0,2009,20.265840,55.176046,14.342611,34.543548,27.572573,1.030764,-2.918897
1,lPxIuEri,Justin Tapia,Hannah Zimmerman,Excellent,cloudy,Flow Traffic,2010-01-05 16:52:16.0000002,16.9,2010-01-05 16:52:16,-1.291824,...,1,1,2010,44.667679,31.832358,23.130775,15.125872,8.755732,8.450134,-0.375217
2,gsVN8JLS,Elizabeth Lopez,Amanda Jackson,Bad,stormy,Congested Traffic,2011-08-18 00:35:00.00000049,5.7,2011-08-18 00:35:00,-1.291242,...,8,3,2011,43.597686,33.712082,19.865289,17.722624,9.847344,1.389525,2.599961
3,9I7kWFgd,Steven Wilson,Amy Horn,Very Good,stormy,Flow Traffic,2012-04-21 04:30:42.0000001,7.7,2012-04-21 04:30:42,-1.291319,...,4,5,2012,42.642965,32.556289,21.063132,15.738963,7.703421,2.799270,0.133905
4,8QN5ZaGN,Alexander Andrews,Cassandra Larson,Bad,stormy,Congested Traffic,2010-03-09 07:51:00.000000135,5.3,2010-03-09 07:51:00,-1.290987,...,3,1,2010,43.329953,39.406828,15.219339,23.732406,15.600745,1.999157,-0.502703


In [6]:
# Remove invalid values
df['fare_amount'] = pd.to_numeric(df['fare_amount'], errors='coerce')
df = df[df['fare_amount'] > 0]
df = df[df['distance'] > 0]
df = df[df['passenger_count'] > 0]

In [7]:
# Handling Missing Values
df.isnull().sum()
(df.isnull().sum() / len(df)) * 100  #percentage of nulls
df.dropna(inplace=True)  # delete nulls becuase there are very few
df.isnull().sum().sum()

np.int64(0)

In [8]:
# Check duplicate rows
df.duplicated().sum()
# there are no duplicates

np.int64(0)

In [9]:
# Remove outliers using IQR

columns = ['fare_amount', 'distance']

for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

     # Capping Outliers
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)

In [10]:
#determine categorical variables
df.select_dtypes(include="object").columns

Index(['User ID', 'User Name', 'Driver Name', 'Car Condition', 'Weather',
       'Traffic Condition', 'key', 'pickup_datetime'],
      dtype='object')

In [11]:
# Encoding Categorical Variables

# Remove unnecessary columns
df.drop(columns=['User ID', 'User Name', 'Driver Name', 'key', 'pickup_datetime'], inplace=True)   #delete categorical variables which not affect

# One-Hot Encoding for Weather becuase it's nominal
df = pd.get_dummies(df, columns=['Weather'], drop_first=True)

# Ordinal Encoding for Traffic Condition because it's ordinal
traffic_map = {
    'Flow Traffic': 0,
    'Dense Traffic': 1,
    'Congested Traffic': 2
}

df['Traffic Condition'] = df['Traffic Condition'].map(traffic_map)

# Ordinal Encoding for Car Condition  because it's ordinal
car_map = {
    'Bad': 0,
    'Good': 1,
    'Very Good': 2,
    'Excellent': 3
}

df['Car Condition'] = df['Car Condition'].map(car_map)

# Check the result
df.head()

,Car Condition,Traffic Condition,fare_amount,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,hour,day,...,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing,Weather_rainy,Weather_stormy,Weather_sunny,Weather_windy
0,2,2,4.5,-1.288826,0.710721,-1.288779,0.710563,1,17,15,...,55.176046,14.342611,34.543548,27.572573,1.030764,-2.918897,False,False,False,True
1,3,0,16.9,-1.291824,0.710546,-1.291182,0.711780,1,16,5,...,31.832358,23.130775,15.125872,8.755732,8.021226,-0.375217,False,False,False,False
2,0,2,5.7,-1.291242,0.711418,-1.291391,0.711231,2,0,18,...,33.712082,19.865289,17.722624,9.847344,1.389525,2.599961,False,True,False,False
3,2,0,7.7,-1.291319,0.710927,-1.291396,0.711363,1,4,21,...,32.556289,21.063132,15.738963,7.703421,2.799270,0.133905,False,True,False,False
4,0,2,5.3,-1.290987,0.711536,-1.290787,0.711811,1,7,9,...,39.406828,15.219339,23.732406,15.600745,1.999157,-0.502703,False,True,False,False


In [12]:
# Split Features and Target
X = df.drop('fare_amount', axis=1)
y = df['fare_amount']

# Split train and testing data
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [13]:
# Feature Selection using RFE
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

# Splitting Features and Target
#X = df.drop('fae_amount', axis=1)
#y = df['fare_amounrt']

rfe_model = LinearRegression()

# Select top 10 features
rfe = RFE(estimator=rfe_model, n_features_to_select=5)

# Fit RFE on training data only
rfe.fit(X_train, y_train)

# Get selected feature names
selected_features = X_train.columns[rfe.support_]

print("Selected Features:")
print(selected_features)

# Apply selected features to both train and test
X_train = X_train[selected_features]
X_test = X_test[selected_features]

# Optional: view feature ranking
feature_ranking = pd.DataFrame({
    'Feature': X.columns,
    'Selected': rfe.support_,
    'Ranking': rfe.ranking_
})

print(feature_ranking.sort_values('Ranking'))

Selected Features:
Index(['pickup_longitude', 'dropoff_longitude', 'dropoff_latitude', 'year',
       'distance'],
      dtype='object')
              Feature  Selected  Ranking
2    pickup_longitude      True        1
4   dropoff_longitude      True        1
5    dropoff_latitude      True        1
11               year      True        1
17           distance      True        1
3     pickup_latitude     False        2
15           sol_dist     False        3
16           nyc_dist     False        4
13           ewr_dist     False        5
18            bearing     False        6
9               month     False        7
10            weekday     False        8
6     passenger_count     False        9
12           jfk_dist     False       10
21      Weather_sunny     False       11
14           lga_dist     False       12
7                hour     False       13
0       Car Condition     False       14
1   Traffic Condition     False       15
20     Weather_stormy     False       16
19

In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
),

"Gradient Boosting": GradientBoostingRegressor(
    n_estimators=100,
    random_state=42
)
}

In [15]:
# from sklearn.linear_model import LinearRegression
# from sklearn.tree import DecisionTreeRegressor
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
#"Random Forest": RandomForestRegressor(random_state=42),
#"Gradient Boosting": GradientBoostingRegressor(random_state=42)
}

In [16]:
results = []

for name, model in models.items():

    print(f"\nTraining {name}...")

    # Create Pipeline
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('model', model)
    ])

    # Train Model
    pipeline.fit(X_train, y_train)

    # Prediction
    y_pred = pipeline.predict(X_test)

    # Evaluation Metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    # Cross Validation
    cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=5,
        scoring='r2'
    )

    cv_mean = cv_scores.mean()

    # Save Results
    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2 Score": r2,
        "CV Score": cv_mean
    })

print("\nTraining Finished!")


Training Linear Regression...

Training Decision Tree...

Training Finished!


In [17]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by="R2 Score", ascending=False)

results_df

,Model,MAE,RMSE,R2 Score,CV Score
0,Linear Regression,1.688838,2.531262,0.784306,0.778317
1,Decision Tree,1.955396,3.117186,0.672893,0.669303


In [18]:
from sklearn.model_selection import GridSearchCV
# Decision Tree Pipeline
dt_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeRegressor(random_state=42))
])

# Hyperparameters
param_grid = {
    'model__max_depth': [10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2]
}

# Grid Search
grid_search = GridSearchCV(
    estimator=dt_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1

)

# Train Grid Search
grid_search.fit(X_train, y_train)

# Best Model
best_dt = grid_search.best_estimator_

# Prediction
y_pred = best_dt.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)


In [19]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree (Before Tuning)",
        "Decision Tree (After Tuning)"
    ],
    "MAE": [
        results_df.loc[results_df["Model"] == "Linear Regression", "MAE"].values[0],
        results_df.loc[results_df["Model"] == "Decision Tree", "MAE"].values[0],
        mae
    ],
    "RMSE": [
        results_df.loc[results_df["Model"] == "Linear Regression", "RMSE"].values[0],
        results_df.loc[results_df["Model"] == "Decision Tree", "RMSE"].values[0],
        rmse
    ],
    "R2 Score": [
        results_df.loc[results_df["Model"] == "Linear Regression", "R2 Score"].values[0],
        results_df.loc[results_df["Model"] == "Decision Tree", "R2 Score"].values[0],
        r2
    ],
    "CV Score": [
        results_df.loc[results_df["Model"] == "Linear Regression", "CV Score"].values[0],
        results_df.loc[results_df["Model"] == "Decision Tree", "CV Score"].values[0],
        grid_search.best_score_
    ]
})

comparison

,Model,MAE,RMSE,R2 Score,CV Score
0,Linear Regression,1.688838,2.531262,0.784306,0.778317
1,Decision Tree (Before Tuning),1.955396,3.117186,0.672893,0.669303
2,Decision Tree (After Tuning),1.522558,2.327183,0.817683,0.814588


In [20]:
comparison_df = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

comparison_df["Error"] = (
    comparison_df["Actual Price"] -
    comparison_df["Predicted Price"]
).abs()

comparison_df.head(20)

,Actual Price,Predicted Price,Error
0,22.25,19.511328,2.738672
1,7.00,6.072786,0.927214
2,10.50,9.168675,1.331325
3,6.90,7.899358,0.999358
4,12.50,10.996083,1.503917
5,8.90,6.449405,2.450595
6,11.70,12.450296,0.750296
7,7.70,8.466850,0.766850
8,22.25,21.912165,0.337835
9,22.25,22.242070,0.007930


In [21]:
import joblib

joblib.dump(grid_search.best_estimator_,open( "Uber_price_model.pkl",'wb'))